In [ ]:
from tensorflow import keras
import numpy as np
from keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
# Load data
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

## Dataset Exploration
Machine learning libraries like Tensorflow or Keras have detailed documentations about pre-defined models, layers, and dataset. These documentations are our best friend to understand a built-in function.

In this task we will try to answer these questions by reading the documentation

### Q1: How many train / test samples are there?

### Q2: What is the data shape of single sample?

### Q3: How are image labels represented in y_train?

### Q4: What is the range of pixel values?

In [ ]:
# Normalize to [0,1]
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

# Add channel dimension: (28, 28) → (28, 28, 1)
x_train = x_train[..., None]
x_test  = x_test[..., None]

### Convolution layer: a powerful tool for image analysis

<img src="images/04_edge_filter_example.png" width="800" />

### Instead of designing filters by hand, we let model learn useful filters from data

In [ ]:
input_layer = layers.Input(shape=(28, 28, 1))
output_layer = layers.Conv2D(32, kernel_size=3, activation="relu")(input_layer)
model = keras.Model(inputs=input_layer, outputs=output_layer)
model.summary()

### Key properties of a convolution layer
Documentation to convolution layer can be found ([here](https://keras.io/api/layers/convolution_layers/convolution2d/))

In [ ]:
# 1. What is are "filters" and "kernel size"

# 2. How to find the total number of parameters in Conv2D?

# 3. Does the number of parameters make sense according to your calculation?

# 4. What does stride control?

# 5. What does padding control?

In [ ]:
# Define a simple CNN

inputs = layers.Input(shape=(28, 28, 1))

x = layers.Conv2D(32, kernel_size=3, activation="relu")(inputs)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation="relu")(x)
outputs = layers.Dense(10, activation="softmax")(x)

model = models.Model(inputs=inputs, outputs=outputs)

In [ ]:
# Compile the model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
# Train the model
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1
)

In [ ]:
history

In [ ]:
sns.lineplot(x=history.epoch, y=history.history['loss'])

In [ ]:
# Evaluation
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

In [ ]:
# Do prediction on test data
y_pred = model.predict(x_test)

# Transform to a dataframe
prediction = pd.DataFrame(y_pred, columns=list(range(10)))
predicted_digits = prediction.idxmax(axis="columns")
print(predicted_digits )

In [ ]:
# Task: Generate classification result confusion matrix
from sklearn.metrics import confusion_matrix

matrix = confusion_matrix(y_test, predicted_digits)
print(matrix)

In [ ]:
confusion_df = pd.DataFrame(matrix, index=range(10), columns=range(10))
sns.heatmap(confusion_df, annot=True, cmap='Blues')

### Take a peak at the "magic" convolution layer outputs:

In [ ]:
# Pick one test sample
test_index = 0
plt.imshow(x_test[test_index, :, :])

In [ ]:
model.summary()

### What does the first convolution layer capture?

In [ ]:
# get intermediate output
def getSubModel(model_full, layer_name):
    return keras.Model(
        inputs=model_full.input,
        outputs=model_full.get_layer(layer_name).output)


sub_model_1 = getSubModel(model, model.layers[1].name)

In [ ]:
sub_model_1.summary()

In [ ]:
first_layer_out = sub_model_1.predict(x_test[test_index:test_index+1, :,:])

In [ ]:
# What do they look like?
fig, axes = plt.subplots(6,6, figsize = (8,8))
axes = axes.flatten()
for ax in axes:
    ax.set_axis_off()
for i in range(32):
    axes[i].imshow(first_layer_out[0, :,:, i])

### What does the second convolution layer capture?

In [ ]:
sub_model_2 = getSubModel(model, model.layers[3].name)

In [ ]:
second_layer_out = sub_model_2(x_test[test_index:test_index+1, :, :])

In [ ]:
second_layer_out.shape

In [ ]:
# What do they look like?
fig, axes = plt.subplots(8,8, figsize = (8,8))
axes = axes.flatten()
for ax in axes:
    ax.set_axis_off()
for i in range(64):
    axes[i].imshow(second_layer_out[0, :,:, i])